In [1]:
import numpy as np
import pandas as pd
from statsmodels.formula.api import ols

# ==========================================
# PASO 1: GENERACIÓN DEL PANEL (6 Buckets x 6 Meses)
# ==========================================
print('--- 1. Generando datos del panel con Tasa y Spread ---')
grupos = [
    "alto_monto_bajo_pd",
    "alto_monto_medio_pd",
    "medio_monto_bajo_pd",
    "medio_monto_medio_pd",
    "bajo_monto_bajo_pd",
    "bajo_monto_alto_pd",
]

meses = ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"]

np.random.seed(42)
registros = []
for grupo in grupos:
    for mes in meses:
        tasa_of = round(np.random.uniform(0.18, 0.28), 4)
        spread = round(np.random.uniform(0.02, 0.08), 4)
        
        conv = round(
            max(
                0.05,
                0.80 - 1.2 * tasa_of - 0.8 * spread + np.random.normal(0, 0.01),
            ),
            4,
        )
        registros.append({
            "grupo": grupo,
            "codmes": mes,
            "tasa_ofertada": tasa_of,
            "spread": spread,
            "conversion": conv,
        })

df = pd.DataFrame(registros)

# Transformaciones logarítmicas base
df["log_conversion"] = np.log(df["conversion"])
df["log_tasa"] = np.log(df["tasa_ofertada"])


# ==========================================
# MÉTODO A: DEMEANING SECUENCIAL + OLS POR BUCKET
# ==========================================
print('\n--- 2A. Demeaning Secuencial (Extrayendo Coeficientes y P-valores) ---')
df_seq = df.copy()

# 1. Demeaning temporal
df_seq["log_conv_dm"] = df_seq["log_conversion"] - df_seq.groupby("codmes")["log_conversion"].transform("mean")
df_seq["log_tasa_dm"] = df_seq["log_tasa"] - df_seq.groupby("codmes")["log_tasa"].transform("mean")
df_seq["spread_dm"] = df_seq["spread"] - df_seq.groupby("codmes")["spread"].transform("mean")

# 2. Demeaning estructural por grupo
df_seq["log_conv_within"] = df_seq["log_conv_dm"] - df_seq.groupby("grupo")["log_conv_dm"].transform("mean")
df_seq["log_tasa_within"] = df_seq["log_tasa_dm"] - df_seq.groupby("grupo")["log_tasa_dm"].transform("mean")
df_seq["spread_within"] = df_seq["spread_dm"] - df_seq.groupby("grupo")["spread_dm"].transform("mean")

resultados_seq = []
for grupo in df_seq["grupo"].unique():
    dg = df_seq[df_seq["grupo"] == grupo].copy()
    m = ols("log_conv_within ~ log_tasa_within + spread_within - 1", data=dg).fit()
    
    resultados_seq.append({
        "grupo": grupo,
        "elast_tasa": m.params["log_tasa_within"],
        "pval_tasa": m.pvalues["log_tasa_within"],
        "sens_spread": m.params["spread_within"],
        "pval_spread": m.pvalues["spread_within"],
        "r2": m.rsquared
    })

df_res_seq = pd.DataFrame(resultados_seq)
print(df_res_seq.to_string(index=False))


# ==========================================
# MÉTODO B: DEMEANING SIMULTÁNEO (Two-Way) + OLS POR BUCKET
# ==========================================
print('\n--- 2B. Demeaning Simultáneo (Extrayendo Coeficientes y P-valores) ---')
df_sim = df.copy()

def demeaning_simultaneo(data, col):
    g_mean = data[col].mean()
    b_mean = data.groupby("grupo")[col].transform("mean")
    t_mean = data.groupby("codmes")[col].transform("mean")
    return data[col] - b_mean - t_mean + g_mean

df_sim["log_conv_dem"] = demeaning_simultaneo(df_sim, "log_conversion")
df_sim["log_tasa_dem"] = demeaning_simultaneo(df_sim, "log_tasa")
df_sim["spread_dem"] = demeaning_simultaneo(df_sim, "spread")

resultados_sim = []
for grupo in df_sim["grupo"].unique():
    dg = df_sim[df_sim["grupo"] == grupo].copy()
    m = ols("log_conv_dem ~ log_tasa_dem + spread_dem - 1", data=dg).fit()
    
    resultados_sim.append({
        "grupo": grupo,
        "elast_tasa": m.params["log_tasa_dem"],
        "pval_tasa": m.pvalues["log_tasa_dem"],
        "sens_spread": m.params["spread_dem"],
        "pval_spread": m.pvalues["spread_dem"],
        "r2": m.rsquared
    })

df_res_sim = pd.DataFrame(resultados_sim)
print(df_res_sim.to_string(index=False))

--- 1. Generando datos del panel con Tasa y Spread ---

--- 2A. Demeaning Secuencial (Extrayendo Coeficientes y P-valores) ---
               grupo  elast_tasa  pval_tasa  sens_spread  pval_spread       r2
  alto_monto_bajo_pd   -0.597547   0.000071    -1.931969     0.000671 0.986234
 alto_monto_medio_pd   -0.624008   0.031330    -1.050597     0.147791 0.819652
 medio_monto_bajo_pd   -0.549905   0.000051    -2.616414     0.009102 0.988420
medio_monto_medio_pd   -0.617036   0.000009    -2.466949     0.000038 0.995289
  bajo_monto_bajo_pd   -0.469447   0.000129    -1.765194     0.030168 0.981644
  bajo_monto_alto_pd   -0.518996   0.000117    -2.010593     0.003178 0.987526

--- 2B. Demeaning Simultáneo (Extrayendo Coeficientes y P-valores) ---
               grupo  elast_tasa  pval_tasa  sens_spread  pval_spread       r2
  alto_monto_bajo_pd   -0.597547   0.000071    -1.931969     0.000671 0.986234
 alto_monto_medio_pd   -0.624008   0.031330    -1.050597     0.147791 0.819652
 medio_mont